# Introduction

<center><h3>**Welcome to the Language modeling Notebook.**</h3></center>

In this assignment, you are going to train a neural network to **generate news headlines**.
To reduce computational needs, we have reduced it to headlines about technology, and a handful of Tech giants.
In this assignment you will:
- Learn to preprocess raw text so it can be fed into an LSTM.
- Make use of the LSTM library of Pytorch, to train a Language model to generate headlines
- Use your network to generate headlines, and judge which headlines are likely or not




**What is a language model?**

Language modeling is the task of assigning a probability to sentences in a language. Besides assigning a probability to each sequence of words, the language models also assigns a probability for the likelihood of a given word (or a sequence of words) to follow a sequence of words.
— Page 105, __[Neural Network Methods in Natural Language Processing](https://www.amazon.com/Language-Processing-Synthesis-Lectures-Technologies/dp/1627052984/)__, 2017.

In terms of neural network, we are training a neural network to produce probabilities (classification) over a fixed vocabulary of words.
Concretely, we are training a neural network to produce:
$$ P ( w_{i+1} | w_1, w_2, w_3, ..., w_i), \forall i \in (1,n)$$

** Why is language modeling important? **

Language modeling is a core problem in NLP.

Language models can either be used as a stand-alone to produce new text that matches the distribution of text the model is trained on, but can also be used at the front-end of a more sophisticated model to produce better results.

Recently for example, the __[BERT](https://arxiv.org/abs/1810.04805)__ paper show-cased that pretraining a large neural network on a language modeling task can help improve state-of-the-art on many NLP tasks.

How good can the generation of a Language model be?

If you have not seen the post about GPT-2 by OpenAI, you should read some of the samples they generated from their language model __[here](https://blog.openai.com/better-language-models/#sample1)__.
Because of computational restrictions, we will not achieve as good text production, but the same algorithm is at the core. They just use more data and compute.

# Library imports

Before starting, make sure you have all these libraries.

In [10]:
# ============ QUICK SETUP: run this ONE cell after every runtime restart ============
!pip -q install segtok
from google.colab import drive
drive.mount('/content/drive')
root_folder = "/content/drive/MyDrive/hw2/"
dataset_folder = root_folder + "dataset/"

import sys, types, importlib
if 'imp' not in sys.modules:   # Python 3.13 fix for autoreload
    imp = types.ModuleType("imp"); imp.reload = importlib.reload; sys.modules["imp"] = imp
%load_ext autoreload
%autoreload 2

import os, json
sys.path.append(root_folder)
from segtok import tokenizer
from collections import Counter
import torch as th
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from utils import validate_to_array

# ---- data ----
with open(dataset_folder+"headline_generation_dataset_processed.json", "r") as f:
    d_released = json.load(f)
with open(dataset_folder+"headline_generation_vocabulary.txt", "r", encoding='utf8') as f:
    vocabulary = f.read().split("\n")
w2i = {w: i for i, w in enumerate(vocabulary)}
i2w = {i: w for i, w in enumerate(vocabulary)}
unkI, padI, startI = w2i['UNK'], w2i['PAD'], w2i['<START>']
vocab_size = len(vocabulary)
input_length = len(d_released[0]['numerized'])
d_train = [d for d in d_released if d['cut'] == 'training']
d_valid = [d for d in d_released if d['cut'] == 'validation']

# ---- helper functions (same as the cells below) ----
def numerize_sequence(tokenized):
    return [w2i.get(w, unkI) for w in tokenized]
def pad_sequence(numerized, pad_index, to_length):
    pad = numerized[:to_length]
    padded = pad + [pad_index] * (to_length - len(pad))
    mask = [w != pad_index for w in padded]
    return padded, mask
def numerized2text(numerized):
    return " ".join(vocabulary[i] for i in numerized if i != padI)
def build_batch(dataset, indices):
    batch = [dataset[i] for i in indices]
    batch_numerized = np.array([a['numerized'] for a in batch])
    start_tokens = np.full((len(batch), 1), startI)
    batch_input = np.concatenate([start_tokens, batch_numerized], axis=1)[:, :-1]
    batch_target = batch_numerized
    batch_target_mask = np.array([a['mask'] for a in batch])
    return batch_input, batch_target, batch_target_mask

# ---- model ----
import language_model
importlib.reload(language_model)
from language_model import LanguageModel

# ---- loss, conversion, device, settings ----
hidden_size, num_layers, dropout = 256, 1, .3
criterion = nn.CrossEntropyLoss(reduction='none')
def loss_fn(pred, target, mask):
    pred = pred.permute(0, 2, 1)
    loss_tensor = criterion(pred, target)
    loss_masked = loss_tensor * mask
    return loss_masked.sum() / mask.sum()
batch_to_torch = lambda b_in,b_target,b_mask: (th.from_numpy(np.asarray(b_in)).long(),
                                               th.from_numpy(np.asarray(b_target)).long(),
                                               th.from_numpy(np.asarray(b_mask)).float())
optimizer_class = optim.Adam
lr, epochs, batch_size = 1e-3, 10, 128
model_id = 'test1'
os.makedirs(root_folder+'models/part1/', exist_ok=True)
device = th.device("cuda" if th.cuda.is_available() else "cpu")
list_to_device = lambda th_obj: [tensor.to(device) for tensor in th_obj]

def evaluate(model, dataset, bs=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0.0
    with th.no_grad():
        for s in range(0, len(dataset), bs):
            b = build_batch(dataset, range(s, min(s+bs, len(dataset))))
            b_in, b_tg, b_mk = list_to_device(batch_to_torch(*b))
            pred = model(b_in)
            total_loss += loss_fn(pred, b_tg, b_mk).item() * b_mk.sum().item()
            total_tokens += b_mk.sum().item()
    model.train()
    return total_loss / total_tokens

print(f"SETUP DONE | train: {len(d_train)} | valid: {len(d_valid)} | device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
SETUP DONE | train: 88568 | valid: 946 | device: cuda


In [5]:
!pip install segtok

Run the first of the following two cells if you are running the homework locally, and run the second cell if you are running the homework in Colab

In [6]:
from google.colab import drive
drive.mount('/content/drive')
root_folder = "/content/drive/MyDrive/hw2/"
dataset_folder = "/content/drive/MyDrive/hw2/dataset/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
DRIVE=False
root_folder = ""
dataset_folder = "dataset/"

In [8]:
import sys, types, importlib
imp = types.ModuleType("imp"); imp.reload = importlib.reload
sys.modules["imp"] = imp
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
sys.path.append(root_folder)
from segtok import tokenizer
from collections import Counter
import torch as th
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import json
from utils import validate_to_array

# Loading the datasets

Make sure the dataset files are all in the `dataset` folder of the assignment.

 - If you are using this notebook locally: You should run the `download_data.sh` script.
 - If you are using the Colab version of the notebook, make sure that your Google Drive is mounted, and you verify from the file explorer in Colab that the files are viewable within `/content/drive/cs182_hw3_public/dataset/`



In [ ]:
# This cell loads the data for the model
# Run this before working on loading any of the additional data

with open(dataset_folder+"headline_generation_dataset_processed.json", "r") as f:
    d_released = json.load(f)

with open(dataset_folder+"headline_generation_vocabulary.txt", "r",encoding='utf8') as f:
    vocabulary = f.read().split("\n")
w2i = {w: i for i, w in enumerate(vocabulary)} # Word to index
i2w = {i: w for i, w in enumerate(vocabulary)} # Index to word
unkI, padI, startI = w2i['UNK'], w2i['PAD'], w2i['<START>']

vocab_size = len(vocabulary)
input_length = len(d_released[0]['numerized']) # The length of the first element in the dataset, they are all of the same length
d_train = [d for d in d_released if d['cut'] == 'training']
d_valid = [d for d in d_released if d['cut'] == 'validation']

print("Number of training samples:",len(d_train))
print("Number of validation samples:",len(d_valid))

Number of training samples: 88568
Number of validation samples: 946


Now that we have loaded the data, let's inspect one of the elements. Each sample in our dataset is has a `numerized` vector, that contains the preprocessed headline. This vector is what we will feed in to the neural network. The field `numerized` corresponds to this list of tokens. The already loaded dictionary `vocabulary` maps token lists to the actual string. Use these elements to recover `title` key of entry 1001 in the training dataset.

**TODO**: Write the numerized2text function in notebook_utils and inspect element 1001 in the training dataset (`entry = d_train[1001]`).



In [ ]:
def numerize_sequence(tokenized):
    return [w2i.get(w, unkI) for w in tokenized]
def pad_sequence(numerized, pad_index, to_length):
    pad = numerized[:to_length]
    padded = pad + [pad_index] * (to_length - len(pad))
    mask = [w != pad_index for w in padded]
    return padded, mask

In [ ]:
def numerized2text(numerized):
    """ Converts an integer sequence in the vocabulary into a string corresponding to the title.

        Arguments:
            numerized: List[int]  -- The list of vocabulary indices corresponding to the string
        Returns:
            title: str -- The string corresponding to the numerized input, without padding.
    """
    #####
    # BEGIN YOUR CODE HERE
    #####
    converted_string = " ".join(vocabulary[i] for i in numerized if i != padI)
    #####
    # END YOUR CODE HERE
    #####

    return converted_string

entry = d_train[1001]
print("Reversing the numerized: "+numerized2text(entry['numerized']))
validate_to_array(numerized2text,(entry['numerized'],),'numerized2text',root_folder)
print("From the `title` entry: "+ entry['title'])

Reversing the numerized: microsoft donates cloud computing ' worth $ 1 bn '
From the `title` entry: Microsoft donates cloud computing 'worth $1 bn'


In language modeling, we train a model to produce the next word in the sequence given all previously generated words. This has, in practice, two steps:


    1. Adding a special <START> token to the start of the sequence for the input. This "shifts" the input to the right by one. We call this the "source" sequence
    2. Making the network predict the original, unshifted version (we call this the "target" sequence)

    
Let's take an example. Say we want to train the network on the sentence: "The cat is great."
The input to the network will be "`<START>` The cat is great." The target will be: "The cat is great".
    
Therefore the first prediction is to select the word "The" given the `<START>` token.
The second prediction is to produce the word "cat" given the two tokens "`<START>` The".
At each step, the network learns to predict the next word, given all previous ones.
    
---

Your next step is to write the build_batch function. Given a dataset, we select a random subset of samples, and will build the "inputs" and the "targets" of the batch, following the procedure we've described.

**TODO**: write the build_batch function. We give you the structure, and you have to fill in where we have left things `your_code`.


In [ ]:
def build_batch(dataset, indices):
    """ Builds a batch of source and target elements from the dataset.

        Arguments:
            dataset: List[db_element] -- A list of dataset elements
            indices: List[int] -- A list of indices of the dataset to sample
        Returns:
            batch_input: List[List[int]] -- List of source sequences
            batch_target: List[List[int]] -- List of target sequences
            batch_target_mask: List[List[int]] -- List of target batch masks
    """
    #####
    # BEGIN YOUR CODE HERE
    #####

    # Recover what the entries for the batch are
    batch = [dataset[i] for i in indices]

    # Get the raw numerized for this input
    batch_numerized = np.array([a['numerized'] for a in batch])

    # Array of startI of shape (batch_size, 1)
    start_tokens = np.full((len(batch), 1), startI)

    # Concatenate start tokens with the input -> [batch_size, sequence_length+1]
    batch_input = np.concatenate([start_tokens, batch_numerized], axis=1)

    # Remove the last word -> [batch_size, sequence_length]
    batch_input = batch_input[:, :-1]

    # The target is the un-shifted numerized input
    batch_target = batch_numerized

    # Mask: 1 for real tokens, 0 for PAD
    batch_target_mask = np.array([a['mask'] for a in batch])

    #####
    # END YOUR CODE HERE
    #####

    return batch_input, batch_target, batch_target_mask

validate_to_array(build_batch,(d_train, range(100)),'build_batch',root_folder)

# Check
bi, bt, bm = build_batch(d_train, range(2))
print("input shape :", bi.shape)
print("target shape:", bt.shape)
print("input [0][:6] :", bi[0][:6])
print("target[0][:6] :", bt[0][:6])

input shape : (2, 20)
target shape: (2, 20)
input [0][:6] : [   0 1958   11    5    1  256]
target[0][:6] : [1958   11    5    1  256 6490]


# Creating the language model

Now that we've written the data pipelining, we are ready to write the Neural network.

The steps to setting up a neural network to do Language modeling are:
- Creating the placeholders for the model, where we can feed in our inputs and targets.
- Creating an RNN of our choice, size, and with optional parameters
- Using the RNN on our placeholder inputs.
- Getting the output from the RNN, and projecting it into a vocabulary sized dimension, so that we can make word predictions.
- Setting up the loss on the outputs so that the network learns to produce the correct words.
- Finally, choosing an optimizer, and defining a training operation: using the optimizer to minimize the loss.

We provide skeleton code for the model, you can fill in the `your_code` section. If you are unfamiliar with Pytorch, we provide some idea of what functions to look for, you should use the Pytorch online documentation.

**TODO**: Fill in the LanguageModel in the language_model file.


In [ ]:
%%writefile /content/drive/MyDrive/hw2/language_model.py
import numpy as np
from segtok import tokenizer
import torch as th
from torch import nn

# Using a basic RNN/LSTM for Language modeling
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, rnn_size, num_layers=1, dropout=0, tie_weights=False):
        super().__init__()

        # Embedding layer of shape [vocab_size, rnn_size]
        self.embedding = nn.Embedding(vocab_size, rnn_size)

        # LSTM layer (batch_first convention)
        self.lstm = nn.LSTM(input_size=rnn_size, hidden_size=rnn_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        # Dropout on the embeddings and on the last hidden output
        self.dropout = nn.Dropout(dropout)
        # Dense layer projecting LSTM outputs to vocabulary logits
        self.output = nn.Linear(rnn_size, vocab_size)
        # Optional weight tying: output layer shares the embedding matrix
        if tie_weights:
            self.output.weight = self.embedding.weight

    def forward(self, x):
        embeds = self.dropout(self.embedding(x))   # [batch, seq, rnn_size]
        lstm_out, _ = self.lstm(embeds)            # [batch, seq, rnn_size]
        lstm_drop = self.dropout(lstm_out)
        logits = self.output(lstm_drop)            # [batch, seq, vocab_size]
        return logits

Overwriting /content/drive/MyDrive/hw2/language_model.py


In [ ]:
import importlib, language_model
importlib.reload(language_model)
from language_model import LanguageModel
import inspect; print(inspect.signature(LanguageModel.__init__))

(self, vocab_size, rnn_size, num_layers=1, dropout=0, tie_weights=False)


# Training the model

Your objective is to train the Language on the dataset you are provided to reach a **validation loss <= 5.50**

**TODO**: Train your model so that it achieves a validation loss of <= 5.5.

**Careful**: we will be testing this loss on an unreleased test set, so make sure to evaluate properly on a validation set and not overfit. You must save the model you want us to test under: models/final_language_model (the .index, .meta and .data files)

**Advice**:
- It should be possible to attain loss <= 5.50 with a 1-layer LSTM of size 256 or less.
- You should not need more than 10 epochs to attain the threshold. More passes over the data can however give you a better model.
- You can however try using:
    - LSTM dropout (Pytorch has a layer for that)
    - Multi-layer RNN cell (Pytorch has a layer for that)
    - Change your optimizers, tune your learning_rate, use a learning rate schedule.
    
**Extra credit**:

Get the loss below **validation loss <= 5.00** and get 5 points of extra-credit on this assignment. Get creative,

but remember, what you do should work on our held-out test set to get the points.

In [ ]:
# We can create our model,
# with parameters of our choosing.
hidden_size = 256
num_layers = 1
dropout = .3

# Cross-entropy loss, averaged only over non-PAD tokens.
criterion = nn.CrossEntropyLoss(reduction='none')
def loss_fn(pred, target, mask):
    pred = pred.permute(0, 2, 1)              # [batch, vocab, seq] -> classes in the middle
    loss_tensor = criterion(pred, target)     # [batch, seq]
    loss_masked = loss_tensor * mask          # ignore PAD tokens
    loss_per_sample = loss_masked.sum() / mask.sum()   # mean over real tokens only
    return loss_per_sample

# numpy -> pytorch: input & target as long, mask as float32
batch_to_torch = lambda b_in,b_target,b_mask: (th.from_numpy(np.asarray(b_in)).long(),
                                               th.from_numpy(np.asarray(b_target)).long(),
                                               th.from_numpy(np.asarray(b_mask)).float())

# Optimizer and training settings
optimizer_class = optim.Adam
lr = 1e-3
epochs = 10
batch_size = 128

model_id = 'test1'
os.makedirs(root_folder+'models/part1/',exist_ok=True)

device = th.device("cuda" if th.cuda.is_available() else "cpu")
print(device)
list_to_device = lambda th_obj: [tensor.to(device) for tensor in th_obj]

cuda


In [ ]:
model = LanguageModel(vocab_size=vocab_size, rnn_size=hidden_size, num_layers=num_layers, dropout=dropout)
optimizer = optimizer_class(model.parameters(), lr=lr)

In [ ]:
# Training loop: trains the model, evaluates on the validation set after
# every epoch, and saves the model with the BEST validation loss.

from tqdm import tqdm

def evaluate(model, dataset, bs=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0.0
    with th.no_grad():
        for s in range(0, len(dataset), bs):
            b = build_batch(dataset, range(s, min(s+bs, len(dataset))))
            b_in, b_tg, b_mk = list_to_device(batch_to_torch(*b))
            pred = model(b_in)
            total_loss += loss_fn(pred, b_tg, b_mk).item() * b_mk.sum().item()
            total_tokens += b_mk.sum().item()
    model.train()
    return total_loss / total_tokens

model.to(device)
model.train()
losses = []
accuracies = []
epoch_train_losses, epoch_val_losses = [], []
best_val = float('inf')

for epoch in range(epochs):
    indices = np.random.permutation(range(len(d_train)))
    t = tqdm(range(0,(len(d_train)//batch_size)+1))
    for i in t:
        idx = indices[i*batch_size:(i+1)*batch_size]
        if len(idx) == 0:
            continue
        # Here is how you obtain a batch:
        batch = build_batch(d_train, idx)
        (batch_input, batch_target, batch_target_mask) = batch_to_torch(*batch)
        (batch_input, batch_target, batch_target_mask) = list_to_device((batch_input, batch_target, batch_target_mask))

        prediction = model(batch_input)
        loss = loss_fn(prediction, batch_target, batch_target_mask)
        losses.append(loss.item())
        accuracy = (th.eq(prediction.argmax(dim=2,keepdim=False),batch_target).float()*batch_target_mask).sum()/batch_target_mask.sum()
        accuracies.append(accuracy.item())

        optimizer.zero_grad()
        loss.backward()
        th.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        if i % 10 == 0:
            t.set_description(f"Epoch: {epoch} Iteration: {i} Loss: {np.mean(losses[-10:]):.3f} Accuracy: {np.mean(accuracies[-10:]):.3f}")

    # ---- validation after each epoch ----
    train_loss = np.mean(losses[-(len(d_train)//batch_size):])
    val_loss = evaluate(model, d_valid)
    epoch_train_losses.append(train_loss)
    epoch_val_losses.append(val_loss)
    print(f"Epoch {epoch}: train loss = {train_loss:.4f} | validation loss = {val_loss:.4f}")

    # save the model only if validation loss improved
    if val_loss < best_val:
        best_val = val_loss
        save_dict = dict(
            kwargs = dict(
                vocab_size=vocab_size,
                rnn_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout
            ),
            model_state_dict = model.state_dict(),
            notes = f"best epoch {epoch}, val loss {val_loss:.4f}",
            optimizer_class = optimizer_class,
            lr = lr,
            epochs = epochs,
            batch_size = batch_size,
        )
        th.save(save_dict,root_folder+f'models/part1/model_{model_id}.pt')
        print("   -> saved (best so far)")

print("Best validation loss:", best_val)

Epoch: 0 Iteration: 690 Loss: 5.943 Accuracy: 0.115: 100%|██████████| 692/692 [00:16<00:00, 40.78it/s]


Epoch 0: train loss = 6.4065 | validation loss = 5.9387
   -> saved (best so far)


Epoch: 1 Iteration: 690 Loss: 5.572 Accuracy: 0.143: 100%|██████████| 692/692 [00:16<00:00, 43.11it/s]


Epoch 1: train loss = 5.6943 | validation loss = 5.5948
   -> saved (best so far)


Epoch: 2 Iteration: 690 Loss: 5.302 Accuracy: 0.159: 100%|██████████| 692/692 [00:16<00:00, 41.81it/s]


Epoch 2: train loss = 5.3770 | validation loss = 5.4174
   -> saved (best so far)


... (10 outputs left out)

Epoch: 8 Iteration: 690 Loss: 4.595 Accuracy: 0.208: 100%|██████████| 692/692 [00:17<00:00, 39.35it/s]


Epoch 8: train loss = 4.6042 | validation loss = 5.1580
   -> saved (best so far)


Epoch: 9 Iteration: 690 Loss: 4.528 Accuracy: 0.213: 100%|██████████| 692/692 [00:17<00:00, 38.81it/s]


Epoch 9: train loss = 4.5287 | validation loss = 5.1532
   -> saved (best so far)
Best validation loss: 5.1531748759086025


In [11]:
# ===== Extra credit attempt: bigger LSTM with weight tying =====
ec_hidden_size = 512
ec_num_layers  = 2
ec_dropout     = 0.5
ec_tie         = True
ec_lr          = 2e-3
ec_epochs      = 15
ec_model_id    = 'ec1'

def evaluate(model, dataset, bs=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0.0
    with th.no_grad():
        for s in range(0, len(dataset), bs):
            b = build_batch(dataset, range(s, min(s+bs, len(dataset))))
            b_in, b_tg, b_mk = list_to_device(batch_to_torch(*b))
            pred = model(b_in)
            total_loss += loss_fn(pred, b_tg, b_mk).item() * b_mk.sum().item()
            total_tokens += b_mk.sum().item()
    model.train()
    return total_loss / total_tokens

ec_model = LanguageModel(vocab_size=vocab_size, rnn_size=ec_hidden_size,
                         num_layers=ec_num_layers, dropout=ec_dropout, tie_weights=ec_tie).to(device)
ec_opt = optim.Adam(ec_model.parameters(), lr=ec_lr)
ec_sched = optim.lr_scheduler.ReduceLROnPlateau(ec_opt, mode='min', factor=0.5, patience=0)

ec_train_losses, ec_val_losses = [], []
ec_best_val, ec_bad_epochs = float('inf'), 0
ec_model.train()
for epoch in range(ec_epochs):
    indices = np.random.permutation(range(len(d_train)))
    t = tqdm(range(0, (len(d_train)//batch_size)+1))
    ep_losses = []
    for i in t:
        idx = indices[i*batch_size:(i+1)*batch_size]
        if len(idx) == 0:
            continue
        b_in, b_tg, b_mk = list_to_device(batch_to_torch(*build_batch(d_train, idx)))
        pred = ec_model(b_in)
        loss = loss_fn(pred, b_tg, b_mk)
        ec_opt.zero_grad()
        loss.backward()
        th.nn.utils.clip_grad_norm_(ec_model.parameters(), 5.0)
        ec_opt.step()
        ep_losses.append(loss.item())
        if i % 10 == 0:
            t.set_description(f"Epoch: {epoch} Loss: {np.mean(ep_losses[-10:]):.3f}")

    val_loss = evaluate(ec_model, d_valid)
    ec_sched.step(val_loss)
    ec_train_losses.append(np.mean(ep_losses))
    ec_val_losses.append(val_loss)
    print(f"Epoch {epoch}: train loss = {np.mean(ep_losses):.4f} | validation loss = {val_loss:.4f} | lr = {ec_opt.param_groups[0]['lr']:.1e}")

    if val_loss < ec_best_val:
        ec_best_val, ec_bad_epochs = val_loss, 0
        ec_save_dict = dict(
            kwargs = dict(vocab_size=vocab_size, rnn_size=ec_hidden_size,
                          num_layers=ec_num_layers, dropout=ec_dropout, tie_weights=ec_tie),
            model_state_dict = ec_model.state_dict(),
            notes = f"extra credit: best epoch {epoch}, val loss {val_loss:.4f}",
            optimizer_class = "Adam", lr = ec_lr, epochs = ec_epochs, batch_size = batch_size,
        )
        th.save(ec_save_dict, root_folder+f'models/part1/model_{ec_model_id}.pt')
        print("   -> saved (best so far)")
    else:
        ec_bad_epochs += 1
        if ec_bad_epochs >= 3:
            print("Early stopping: no improvement for 3 epochs")
            break

print("Extra credit best validation loss:", ec_best_val)

Epoch: 0 Loss: 6.418: 100%|██████████| 692/692 [00:35<00:00, 19.39it/s]


Epoch 0: train loss = 6.8062 | validation loss = 6.1385 | lr = 2.0e-03
   -> saved (best so far)


Epoch: 1 Loss: 6.194: 100%|██████████| 692/692 [00:36<00:00, 19.06it/s]


Epoch 1: train loss = 6.2653 | validation loss = 5.8769 | lr = 2.0e-03
   -> saved (best so far)


Epoch: 2 Loss: 5.996: 100%|██████████| 692/692 [00:38<00:00, 18.17it/s]


Epoch 2: train loss = 6.0538 | validation loss = 5.7234 | lr = 2.0e-03
   -> saved (best so far)


... (20 outputs left out)

Epoch: 13 Loss: 5.257: 100%|██████████| 692/692 [00:38<00:00, 17.80it/s]


Epoch 13: train loss = 5.2952 | validation loss = 5.1753 | lr = 2.0e-03
   -> saved (best so far)


Epoch: 14 Loss: 5.315: 100%|██████████| 692/692 [00:38<00:00, 17.77it/s]


Epoch 14: train loss = 5.2609 | validation loss = 5.1448 | lr = 2.0e-03
   -> saved (best so far)
Extra credit best validation loss: 5.1447943823318685


In [12]:
# ===== Extra credit (continued): resume training the same model with lower dropout =====
if 'ec_model' not in globals():   # if the runtime restarted, reload the saved ec1 model
    _sd = th.load(root_folder+'models/part1/model_ec1.pt', map_location='cpu', weights_only=False)
    ec_model = LanguageModel(**_sd['kwargs'])
    ec_model.load_state_dict(_sd['model_state_dict'])
    ec_best_val = float(_sd['notes'].split('val loss ')[-1])
    ec_train_losses, ec_val_losses = [], []
ec_model.to(device)

# lower the dropout so the model learns faster
ec_dropout = 0.3
ec_model.dropout.p = ec_dropout
ec_model.lstm.dropout = ec_dropout

ec_opt = optim.Adam(ec_model.parameters(), lr=ec_lr)
ec_sched = optim.lr_scheduler.ReduceLROnPlateau(ec_opt, mode='min', factor=0.5, patience=0)
extra_epochs, ec_bad_epochs = 20, 0
start_ep = len(ec_val_losses)
print("Starting from best validation loss:", round(ec_best_val, 4))

ec_model.train()
for epoch in range(start_ep, start_ep + extra_epochs):
    indices = np.random.permutation(range(len(d_train)))
    t = tqdm(range(0, (len(d_train)//batch_size)+1))
    ep_losses = []
    for i in t:
        idx = indices[i*batch_size:(i+1)*batch_size]
        if len(idx) == 0:
            continue
        b_in, b_tg, b_mk = list_to_device(batch_to_torch(*build_batch(d_train, idx)))
        loss = loss_fn(ec_model(b_in), b_tg, b_mk)
        ec_opt.zero_grad()
        loss.backward()
        th.nn.utils.clip_grad_norm_(ec_model.parameters(), 5.0)
        ec_opt.step()
        ep_losses.append(loss.item())
        if i % 10 == 0:
            t.set_description(f"Epoch: {epoch} Loss: {np.mean(ep_losses[-10:]):.3f}")

    val_loss = evaluate(ec_model, d_valid)
    ec_sched.step(val_loss)
    ec_train_losses.append(np.mean(ep_losses))
    ec_val_losses.append(val_loss)
    print(f"Epoch {epoch}: train loss = {np.mean(ep_losses):.4f} | validation loss = {val_loss:.4f} | lr = {ec_opt.param_groups[0]['lr']:.1e}")

    if val_loss < ec_best_val:
        ec_best_val, ec_bad_epochs = val_loss, 0
        ec_save_dict = dict(
            kwargs = dict(vocab_size=vocab_size, rnn_size=ec_hidden_size,
                          num_layers=ec_num_layers, dropout=ec_dropout, tie_weights=ec_tie),
            model_state_dict = ec_model.state_dict(),
            notes = f"extra credit: best epoch {epoch}, val loss {val_loss:.4f}",
            optimizer_class = "Adam", lr = ec_lr, epochs = epoch+1, batch_size = batch_size,
        )
        th.save(ec_save_dict, root_folder+f'models/part1/model_{ec_model_id}.pt')
        print("   -> saved (best so far)")
    else:
        ec_bad_epochs += 1
        if ec_bad_epochs >= 3:
            print("Early stopping: no improvement for 3 epochs")
            break

print("Extra credit best validation loss:", ec_best_val)

Starting from best validation loss: 5.1448


Epoch: 15 Loss: 4.949: 100%|██████████| 692/692 [00:40<00:00, 17.20it/s]


Epoch 15: train loss = 4.9730 | validation loss = 5.0743 | lr = 2.0e-03
   -> saved (best so far)


Epoch: 16 Loss: 4.848: 100%|██████████| 692/692 [00:38<00:00, 17.94it/s]


Epoch 16: train loss = 4.8868 | validation loss = 5.0440 | lr = 2.0e-03
   -> saved (best so far)


Epoch: 17 Loss: 4.849: 100%|██████████| 692/692 [00:39<00:00, 17.63it/s]


... (23 outputs left out)

Epoch: 29 Loss: 4.284: 100%|██████████| 692/692 [00:38<00:00, 17.90it/s]


Epoch 29: train loss = 4.2929 | validation loss = 4.9330 | lr = 2.5e-04


Epoch: 30 Loss: 4.235: 100%|██████████| 692/692 [00:38<00:00, 17.88it/s]


Epoch 30: train loss = 4.2628 | validation loss = 4.9376 | lr = 1.3e-04
Early stopping: no improvement for 3 epochs
Extra credit best validation loss: 4.923880280919943


# Using the language model

Congratulations, you have now trained a language model! We can now use it to evaluate likely news headlines, as well as generate our very own headlines.

**TODO**: Complete the three parts below, using the model you have trained.

## (1) Evaluation loss

To evaluate the language model, we evaluate its loss (ability to predict) on unseen data that is reserved for evaluation.
Your first evaluation is to load the model you trained, and obtain a test loss. If you are running this validation and not training, run the setup cell above the training loop first.

In [13]:
model_id = "ec1"
save_dict = th.load(root_folder+'models/part1/'+f"model_{model_id}.pt",map_location='cpu', weights_only=False)
model = LanguageModel(**save_dict['kwargs'])
model.load_state_dict(save_dict['model_state_dict'])
model.eval()

LanguageModel(
  (embedding): Embedding(10000, 512)
  (lstm): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (output): Linear(in_features=512, out_features=10000, bias=True)
)

In [14]:
# We will evaluate your model in the best_models folder
# In a very similar way as the code below.
# Make sure your validation loss is below the threshold we specified
# and that you didn't train using the validation set

batch = build_batch(d_valid, range(len(d_valid)))
(batch_input, batch_target, batch_target_mask) = batch_to_torch(*batch)
prediction = model(batch_input.long())
loss = loss_fn(prediction, batch_target, batch_target_mask)
print("Evaluation set loss:", loss.item())

Evaluation set loss: 4.9238715171813965


In [2]:
# Validation loss AND next-word accuracy (non-PAD tokens) of both Part 1 models
def eval_loss_acc(model_id):
    sd = th.load(root_folder+'models/part1/'+f"model_{model_id}.pt", map_location='cpu', weights_only=False)
    m = LanguageModel(**sd['kwargs']); m.load_state_dict(sd['model_state_dict']); m.eval()
    b_in, b_tg, b_mk = batch_to_torch(*build_batch(d_valid, range(len(d_valid))))
    with th.no_grad():
        pred = m(b_in)
        loss = loss_fn(pred, b_tg, b_mk).item()
        acc = ((pred.argmax(dim=2) == b_tg).float() * b_mk).sum().item() / b_mk.sum().item()
    return loss, acc

for mid, name in [("test1", "Baseline (LSTM-256)"), ("ec1", "Best / extra credit (LSTM-512)")]:
    l, a = eval_loss_acc(mid)
    print(f"{name:32s} validation loss {l:.4f} | validation next-word accuracy {a*100:.1f}%")

Baseline (LSTM-256)              validation loss 5.1532 | validation next-word accuracy 17.8%
Best / extra credit (LSTM-512)   validation loss 4.9239 | validation next-word accuracy 20.8%


In [15]:
# Your best performing model should go here.
os.makedirs(root_folder+"best_models",exist_ok=True)
best_model_file = root_folder+"best_models/part1_best_model.pt"

# store optimizer as a plain name so the file loads with default torch.load settings
save_dict_best = dict(save_dict)
if not isinstance(save_dict_best['optimizer_class'], str):
    save_dict_best['optimizer_class'] = save_dict_best['optimizer_class'].__name__
th.save(save_dict_best, best_model_file)

# check: reload with default settings and re-evaluate
chk = th.load(best_model_file, map_location='cpu')
m_chk = LanguageModel(**chk['kwargs']); m_chk.load_state_dict(chk['model_state_dict']); m_chk.eval()
b = batch_to_torch(*build_batch(d_valid, range(len(d_valid))))
with th.no_grad():
    print("Saved:", best_model_file)
    print("kwargs:", chk['kwargs'])
    print("Reloaded best model validation loss:", loss_fn(m_chk(b[0]), b[1], b[2]).item())

Saved: /content/drive/MyDrive/hw2/best_models/part1_best_model.pt
kwargs: {'vocab_size': 10000, 'rnn_size': 512, 'num_layers': 2, 'dropout': 0.3, 'tie_weights': True}
Reloaded best model validation loss: 4.9238715171813965


## (2) Evaluation of likelihood of data

One use of a language model is to see what data is more likely to have originated from the training data. Because we have trained our model on news headlines, we can see which of these headlines is more likely:

``Apple to release another iPhone in September``


 ``Apple and Samsung resolve all lawsuits amicably``

**TODO**: Use the model to obtain the loss the neural network assigns to each sentence.
Because the neural network assigns probability to the words appearing in a sequence, this loss can be used as a proxy to measure how likely the sentence is to have occurred in the dataset.
Once you have the loss for each headline, write down which sentence was judged to be more likely, and explain why/if you think this is coherent.

**Your answer:** Using the best model (2-layer LSTM, validation loss 4.92), the first headline, "Apple to release new iPhone in July", has a loss of 2.75, while the second headline, "Apple and Samsung resolve all lawsuits", has a loss of 4.99. The model therefore judges the first headline to be much more likely. (The first, smaller model gave the same ranking: 3.07 vs 5.74.)

This is coherent. The dataset is made of tech news headlines, where headlines about Apple releasing a new iPhone in a given month are very common, so every word in the first headline ("release", "new", "iphone", "in", "july") is highly predictable from the words before it. The second headline describes an unusual event: Apple and Samsung were famous for their long patent war, so headlines about them "resolving all lawsuits" are rare in the training data, and the model assigns low probability to words like "resolve" following "apple and samsung". A lower loss means a higher probability under the language model, so the ranking matches what we would expect from real tech news.


In [16]:
def raw_sample_pred(headline, model):
    #####

    # Tokenize the headline (same as the Preprocessing section)
    tokenized = tokenizer.word_tokenizer(headline.lower())

    # Numerize the tokenized headline
    numerized = numerize_sequence(tokenized)

    # Pad to input_length and obtain the mask
    padded, mask = pad_sequence(numerized, padI, input_length)

    # Input = <START> + headline (shifted right), target = headline
    input_headline = th.LongTensor([[startI] + padded[:-1]])
    with th.no_grad():
        pred_headline = model(input_headline)
    target_headline = th.LongTensor([padded])
    mask = th.FloatTensor([mask])

    #####


    return pred_headline,target_headline,mask

In [17]:
model.eval()

headline1 = "Apple to release new iPhone in July"
headline2 = "Apple and Samsung resolve all lawsuits"

headlines = [headline1.lower(), headline2.lower()] # Our LSTM is trained on lower-cased headlines
for headline in headlines:
    pred_headline,target_headline,mask = raw_sample_pred(headline, model)
    loss = loss_fn(pred_headline, target_headline, mask).item() # Obtain the loss

    print("----------------------------------------")
    print("Headline:", headline)
    print("Loss of the headline:", loss)

# Save outputs to submission_logs (same as validate_to_array, but works with numpy 2)
_outs = [tuple(t.detach().cpu().numpy() for t in raw_sample_pred(h, model)) for h in headlines]
_arr = np.empty(len(_outs), dtype=object)
for _k, _o in enumerate(_outs):
    _arr[_k] = _o
np.save(os.path.join(root_folder, 'submission_logs', 'raw_sample_pred'), _arr, allow_pickle=True)
# Important check: one headline should be more likely (and have lower loss)
# Than the other headline. You should know which headline should have lower loss.

----------------------------------------
Headline: apple to release new iphone in july
Loss of the headline: 2.7505078315734863
----------------------------------------
Headline: apple and samsung resolve all lawsuits
Loss of the headline: 4.989663600921631


## (3) Generation of headlines

We can use our language model to generate text according to the distribution of our training data.
The way generation works is the following:

We seed the model with a beginning of sequence, and obtain the distribution for the next word.
We select the most likely word (argmax) and add it to our sequence of words.
Now our sequence is one word longer, and we can feed it in again as an input, for the network to produce the next sentence.
We do this a fixed number of times (up to 20 words), and obtain automatically generated headlines!


We have provided a few headline starters that should produce interesting generated headlines.

**TODO:** Get creative and find at least 2 more headline_starters that produce interesting headlines.

In [18]:
def generate_sentence(headline_starter, model):
    # Tokenize and numerize the headline. Put the numerized headline
    # beginning in `current_build`
    tokenized = tokenizer.word_tokenizer(headline_starter.lower())
    current_build = [startI] + numerize_sequence(tokenized)

    while len(current_build) < input_length:
        # Pad the current_build into a input_length vector.
        current_padded, _m = pad_sequence(current_build, padI, input_length)

        # Obtain the logits for the current padded sequence
        with th.no_grad():
            logits = model(th.LongTensor([current_padded]))
        logits_np = logits.detach().cpu().numpy()

        # The row of logits for the last non-pad input
        last_logits = logits_np[0, len(current_build) - 1].copy()
        last_logits[unkI] = -np.inf   # never generate the unknown-word token

        # Pick the highest scoring word (greedy / argmax) and append it
        current_build.append(int(np.argmax(last_logits)))

    # Convert word indices back to a string (skip <START> and PAD)
    produced_sentence = " ".join(vocabulary[i] for i in current_build[1:] if i != padI)
    return produced_sentence

In [19]:
model.eval()
# Here are some headline starters.
# They're all about tech companies, because
# That is what is in our dataset
headline_starters = ["apple has released", "google has released", "amazon", "tesla to",
                     # --- my own headline starters ---
                     "microsoft to buy", "facebook users", "samsung galaxy", "twitter ceo"]
for headline_starter in headline_starters:
    print("===================")
    print("Generating headline starting with: "+headline_starter)

    produced_sentence = generate_sentence(headline_starter, model)
    print(produced_sentence)
validate_to_array(generate_sentence,zip(headline_starters,[model]*len(headline_starters)),"generate_sentence",root_folder,multi=True)

Generating headline starting with: apple has released
apple has released a new version of the iphone x - and it could be a big deal for
Generating headline starting with: google has released
google has released a new version of its new android app that lets you share your phone with a
Generating headline starting with: amazon
amazon to open first physical store in nyc , and the new york times , and it's a huge
Generating headline starting with: tesla to
tesla to buy apple stake in $ 1 billion in sharp , nikkei says says it will open to
Generating headline starting with: microsoft to buy
microsoft to buy linkedin for $ 26.2 billion in cash , wsj says says it will be ' a
Generating headline starting with: facebook users
facebook users are ' like ' to be a ' dislike ' button for the first time in the
Generating headline starting with: samsung galaxy
samsung galaxy s4 emerges to apple in china , but it may be a big blow to the iphone
Generating headline starting with: twitter ceo
twitter c

## All done

You are done with the first part of the HW.

Next notebook deals with Summarization of text!


# Preprocessing (read only)


**You can skip this section, however you may find these functions useful in later sections of this notebook**

We have provided this code so you see how the dataset was generated. You will have to come back some of these functions later in the assignment, so feel free to read through, to get familiar.

In [ ]:
# You do not need to run this
# This is to show you how the dataset was created
# You should read to understand, so you can preprocess text
# In the same way, in the evaluation section

for a in dataset:
    a['tokenized'] = tokenizer.word_tokenizer(a['title'].lower())

In [ ]:
# You do not need to run this
# This is to show you how the dataset was created
# You should read to understand, so you can preprocess text
# In the same way, in the evaluation section

word_counts = Counter()
for a in dataset:
    word_counts.update(a['tokenized'])

print(word_counts.most_common(30))

In [ ]:
# You do not need to run this
# This is to show you how the dataset was created
# You should read to understand, so you can preprocess text
# In the same way, in the evaluation section

# Creating the vocab
vocab_size = 20000
special_words = ["<START>", "UNK", "PAD"]
vocabulary = special_words + [w for w, c in word_counts.most_common(vocab_size-len(special_words))]
w2i = {w: i for i, w in enumerate(vocabulary)}

# Numerizing and padding
input_length = 20
unkI, padI, startI = w2i['UNK'], w2i['PAD'], w2i['<START>']

for a in dataset:
    a['numerized'] = numerize_sequence(a['tokenized']) # Change words to IDs
    a['numerized'], a['mask'] = pad_sequence(a['numerized'], padI, input_length) # Append appropriate PAD tokens

# Compute fraction of words that are UNK:
word_counters = Counter([w for a in dataset for w in a['input'] if w != padI])

print("Fraction of UNK words:", float(word_counters[unkI]) / sum(word_counters.values()))

In [ ]:
# You do not need to run this
# This is to show you how the dataset was created
# You should read to understand, so you can preprocess text
# In the same way, in the evaluation section

d_released_processed   = [d for d in dataset if d['cut'] != 'testing']
d_unreleased_processed = [d for d in dataset if d['cut'] == 'testing']

with open("dataset/headline_generation_dataset_processed.json", "w") as f:
    json.dump(d_released_processed, f)

# This file is purposefully left out of the assignment, we will use it to evaluate your model.
with open("dataset/headline_generation_dataset_unreleased_processed.json", "w") as f:
    json.dump(d_unreleased_processed, f)

with open("dataset/headline_generation_vocabulary.txt", "w") as f:
    f.write("\n".join(vocabulary).encode('utf8'))